In [31]:
import os
import pandas as pd
from datasets import Dataset

from utils.bq import get_client, query_to_df
from utils.hub_card import push_dataset_card
from utils.clean_cohort_df import simplify_race_column

client, PROJECT_NAME = get_client()

DESTINATION = os.environ.get('COHORT_DESTINATION')
if not DESTINATION:
    raise ValueError("Need to run cohort_base.py script first to save the cohort data in big query, then you can use this notebook")

In [32]:
# Load base ED patient cohort from BigQuery (built by cohort_base.py).
# One row per ED visit. Retains all discharge_location values at this stage,
# including AGAINST ADVICE -- those are removed in the next step.

df_cohort = query_to_df(client, f"SELECT * FROM `{DESTINATION}`")
print(f"Shape: {df_cohort.shape}")
print(f"--- Cohort Label Distribution ---")
print(df_cohort["cohort_label"].value_counts())
df_cohort.head()

Shape: (399573, 22)
--- Cohort Label Distribution ---
cohort_label
ED_DISCHARGE_STABLE        241215
ED_WARD_DISCHARGE          126879
ED_DIRECT_ICU               21817
ED_WARD_ICU                  9295
ED_DISCHARGE_RETURN_ICU       295
ED_DISCHARGE_DIED_72H          72
Name: count, dtype: int64


,subject_id,ed_stay_id,hadm_id,ed_intime,ed_outtime,disposition,race,arrival_transport,first_careunit,first_icu_intime,base_pathway,cohort_label,gender,anchor_age,anchor_year,admittime,dischtime,admission_type,discharge_location,insurance,language,marital_status
0,10002013,34931809,21763296,2165-11-22 20:54:00,2165-11-23 08:20:42,ADMITTED,WHITE,WALK IN,Medical Intensive Care Unit (MICU),NaT,ED_WARD_DISCHARGE,ED_WARD_DISCHARGE,F,53,2156,2165-11-23 08:19:00,2165-11-26 15:40:00,DIRECT EMER.,HOME HEALTH CARE,Medicaid,English,SINGLE
1,10014354,38849383,27562275,2148-07-17 20:36:00,2148-07-18 04:46:00,ADMITTED,WHITE,AMBULANCE,Cardiology Surgery Intermediate,NaT,ED_WARD_DISCHARGE,ED_WARD_DISCHARGE,M,60,2146,2148-07-18 02:31:00,2148-07-20 17:47:00,DIRECT EMER.,HOME HEALTH CARE,Medicare,English,SINGLE
2,10018862,35211473,22949345,2149-05-01 17:05:00,2149-05-01 22:22:00,ADMITTED,WHITE,WALK IN,Transplant,NaT,ED_WARD_DISCHARGE,ED_WARD_DISCHARGE,F,56,2148,2149-05-01 21:43:00,2149-05-05 18:20:00,DIRECT EMER.,HOME,Medicaid,English,MARRIED
3,10019003,38020791,26529390,2155-05-17 21:03:00,2155-05-18 00:03:15,ADMITTED,WHITE,WALK IN,Hematology/Oncology Intermediate,NaT,ED_WARD_DISCHARGE,ED_WARD_DISCHARGE,F,65,2148,2155-05-17 00:02:00,2155-05-19 18:27:00,DIRECT EMER.,HOME,Medicare,English,WIDOWED
4,10021395,31017572,21372240,2132-07-30 18:58:00,2132-07-31 04:56:00,ADMITTED,WHITE - OTHER EUROPEAN,WALK IN,Medicine,NaT,ED_WARD_DISCHARGE,ED_WARD_DISCHARGE,F,86,2128,2132-07-31 03:28:00,2132-07-31 15:09:00,DIRECT EMER.,HOME,Medicare,English,WIDOWED


In [36]:
# Fix cohort label mismatches caused by MIMIC data inconsistencies between
# edstays.disposition and the actual admissions/transfers records.
#
# Case 1: Labeled ED_DISCHARGE_* but has hadm_id + admittime → patient was actually admitted
#   Sub-case: has first_icu_intime → ED_DIRECT_ICU or ED_WARD_ICU (based on first_careunit)
#   Sub-case: no ICU → ED_WARD_DISCHARGE
#
# Case 2: Labeled ED_WARD_DISCHARGE but no hadm_id / admittime → patient was not admitted
#   → reclassify as ED_DISCHARGE_STABLE

discharge_labels = ['ED_DISCHARGE_STABLE', 'ED_DISCHARGE_RETURN_ICU', 'ED_DISCHARGE_DIED_72H']

icu_units = [
    'Medical Intensive Care Unit (MICU)', 'Surgical Intensive Care Unit (SICU)',
    'Cardiac Vascular Intensive Care Unit (CVICU)', 'Medical/Surgical Intensive Care Unit (MICU/SICU)',
    'Coronary Care Unit (CCU)', 'Trauma SICU (TSICU)',
    'Neuro Surgical Intensive Care Unit (Neuro SICU)', 'Neuro Intermediate'
]

# Case 1: discharge label but has an actual admission record
admitted_mask = (
    df_cohort['cohort_label'].isin(discharge_labels) &
    df_cohort['hadm_id'].notna() &
    df_cohort['admittime'].notna()
)
print(f"Case 1 (discharge label but admitted): {admitted_mask.sum()} rows")

def reclassify_admitted(row):
    if pd.notna(row['first_icu_intime']):
        return 'ED_DIRECT_ICU' if row['first_careunit'] in icu_units else 'ED_WARD_ICU'
    return 'ED_WARD_DISCHARGE'

df_cohort.loc[admitted_mask, 'cohort_label'] = df_cohort[admitted_mask].apply(reclassify_admitted, axis=1)

# Case 2: ward label but no admission record
not_admitted_mask = (
    (df_cohort['cohort_label'] == 'ED_WARD_DISCHARGE') &
    df_cohort['hadm_id'].isna() &
    df_cohort['admittime'].isna()
)
print(f"Case 2 (ward label but no admission): {not_admitted_mask.sum()} rows")
df_cohort.loc[not_admitted_mask, 'cohort_label'] = 'ED_DISCHARGE_STABLE'

print("\nUpdated cohort label distribution:")
print(df_cohort['cohort_label'].value_counts())

Case 1 (discharge label but admitted): 36492 rows
Case 2 (ward label but no admission): 384 rows

Updated cohort label distribution:
cohort_label
ED_DISCHARGE_STABLE        205206
ED_WARD_DISCHARGE          162870
ED_DIRECT_ICU               21894
ED_WARD_ICU                  9335
ED_DISCHARGE_RETURN_ICU       213
ED_DISCHARGE_DIED_72H          55
Name: count, dtype: int64


In [37]:
# Remove AGAINST ADVICE discharges.
# These represent patient-driven departures (left AMA, eloped) rather than
# provider disposition decisions, so they're out of scope for the RL policy.
advice_df = df_cohort[df_cohort['discharge_location'] != 'AGAINST ADVICE'].copy()
print(f"Dropped {len(df_cohort) - len(advice_df):,} AGAINST ADVICE rows")
print(f"Remaining: {len(advice_df):,}")

Dropped 1,563 AGAINST ADVICE rows
Remaining: 398,010


In [38]:
# Collapse consecutive ED visits that share a hadm_id into a single row.
# Some patients have two back-to-back ED stays under one admission (e.g. brief
# discharge and immediate return). We keep the first stay's identifiers and
# merge the second stay's ed_outtime/disposition/cohort_label into it,
# storing the second ed_stay_id in ed_stay_id_2.
dup_counts = advice_df['hadm_id'].value_counts()
dup_hadm_ids = dup_counts[dup_counts == 2].index

singles = advice_df[~advice_df['hadm_id'].isin(dup_hadm_ids)].copy()
dups = advice_df[advice_df['hadm_id'].isin(dup_hadm_ids)].sort_values(['hadm_id', 'ed_intime'])

first_rows = dups.groupby('hadm_id').nth(0).reset_index(drop=True)
second_rows = dups.groupby('hadm_id').nth(1).reset_index(drop=True)

merged_dups = first_rows.copy()
merged_dups['ed_outtime'] = second_rows['ed_outtime'].values
merged_dups['disposition'] = second_rows['disposition'].values
merged_dups['cohort_label'] = second_rows['cohort_label'].values
merged_dups['ed_stay_id_2'] = second_rows['ed_stay_id'].values
merged_dups.drop(columns=['base_pathway'], inplace=True)

singles['ed_stay_id_2'] = float('nan')
singles.drop(columns=['base_pathway'], inplace=True)

cohort_df = pd.concat([singles, merged_dups], ignore_index=True)

print(f"advice_df rows: {len(advice_df):,}")
print(f"cohort_df rows: {len(cohort_df):,}  (expected {len(advice_df) - len(dup_hadm_ids):,})")
print(f"Max hadm_id count: {cohort_df['hadm_id'].value_counts().max()}  (expected 1)")

advice_df rows: 398,010
cohort_df rows: 397,675  (expected 397,675)
Max hadm_id count: 1  (expected 1)


In [ ]:
# Drop the rows in this subset where admittime is nan, switch the rest of stay window end for the ed_outtime value instead.  Put in middle in cell below
cohort_df[(cohort_df['stay_window_start']) > (cohort_df['stay_window_end'])].index

In [ ]:
# Add stay window columns: full time span from ED arrival to final discharge.
# For admitted patients: ed_intime → dischtime
# For ED-only patients: ed_intime → ed_outtime
cohort_df['stay_window_start'] = pd.to_datetime(cohort_df['ed_intime'])
cohort_df['stay_window_end'] = pd.to_datetime(
    cohort_df['dischtime'].fillna(cohort_df['ed_outtime'])
)

# Add ED boarding time: hours spent in ED after admission decision was made.
# Calculated as ed_outtime - admittime (null for ED-only discharges).
cohort_df['ed_boarding_time_min'] = (
    (pd.to_datetime(cohort_df['ed_outtime']) - pd.to_datetime(cohort_df['admittime']))
    .dt.total_seconds()
    / 60
)

print(f"Patients with boarding time (admitted):   {cohort_df['ed_boarding_time_hrs'].notna().sum():,}")
print(f"Patients without boarding time (ED-only): {cohort_df['ed_boarding_time_hrs'].isna().sum():,}")

Patients with boarding time (admitted):   192,201
Patients without boarding time (ED-only): 205,474


In [ ]:
cohort_df = simplify_race_column(cohort_df)
cohort_df['race'].value_counts(dropna=False)

race
White              233140
Black               86687
Hispanic            33410
Other               25335
Asian               18122
Native American       981
Name: count, dtype: int64

In [ ]:
import math

time_block = 30

# Recalculate time_steps: number of  windows in each patient stay
total = pd.to_datetime(cohort_df['stay_window_end']) - pd.to_datetime(cohort_df['stay_window_start'])
cohort_df['time_steps'] = total.dropna().apply(lambda x: math.ceil(x.total_seconds() / 60 / time_block)) + 1

# Drop records with no hospital stay (null or non-positive window)
cohort_df = cohort_df[cohort_df['time_steps'].notna() & (cohort_df['time_steps'] > 0)].copy()
print(f"Rows after dropping invalid stay windows: {len(cohort_df)}")

In [ ]:
# Sort before expanding so each stay's rows are contiguous after repeat
new_df = (
    cohort_df[['ed_stay_id', 'subject_id', 'hadm_id', 'time_steps']]
    .sort_values(by=['subject_id', 'ed_stay_id', 'hadm_id'])
    .reset_index(drop=True)
)

new_df = new_df.loc[new_df.index.repeat(new_df['time_steps'])].reset_index(drop=True)

new_df.head(10)

In [ ]:
row_counts = new_df.groupby('ed_stay_id').size().reset_index(name='row_count')
row_counts = row_counts.merge(cohort_df[['ed_stay_id', 'time_steps']].drop_duplicates(), on='ed_stay_id')
mismatched = row_counts[row_counts['row_count'] != row_counts['time_steps']]
print(f"Mismatched stays: {len(mismatched)}")
mismatched

In [ ]:
DESCRIPTION = (
    "Processed ED patient cohort derived from MIMIC-IV. One row per ED visit (ed_stay_id). "
    "Built from cohort_base (BigQuery) with the following post-processing steps: "
    "(1) AGAINST ADVICE discharge_location records removed — patient-driven departures are out of scope. "
    "(2) Consecutive ED visits sharing a hadm_id collapsed into single rows; second ed_stay_id stored in ed_stay_id_2. "
    "(3) stay_window_start/stay_window_end columns added covering ED arrival to final discharge. "
    "(4) ed_boarding_time_hrs added: hours in ED after admission decision (null for ED-only patients). "
    "Intended as the primary cohort table for feature engineering and RL state construction."
)

ds = Dataset.from_pandas(cohort_df)
ds.push_to_hub("ADS599-Capstone/raw_data", config_name="cohort_full", split='cohort_full', data_dir="cohort")
print("Pushed cohort to HuggingFace Hub.")

In [47]:
import pickle

stay_id_remap = (
    cohort_df[cohort_df['ed_stay_id_2'].notna()]
    .set_index('ed_stay_id_2')['ed_stay_id']
    .astype(int)
    .to_dict()
)

with open("stay_id_remap.pkl", "wb") as f:
    pickle.dump(stay_id_remap, f)

print(f"Saved stay_id_remap with {len(stay_id_remap):,} entries to stay_id_remap.pkl")

Saved stay_id_remap with 335 entries to stay_id_remap.pkl
